## Load necessary libraries

In [ ]:
#importing the libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import time
from datetime import datetime
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from torchsummary import summary
from scipy.io import savemat
import os
from scipy.stats import multivariate_normal
from scipy.integrate import dblquad

In [ ]:
# Device will determine whether to run the training on GPU or CPU.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
display(device)
torch.cuda.is_available() 

## Import the training dataset

In [ ]:
relative_folder = '../../../Data/Datasets/2. Sensor 2/Training Dataset/'
abs_folder = os.path.abspath(relative_folder)
files = os.listdir(abs_folder)
target_file = files[0] 
full_path = os.path.join(abs_folder, target_file)
data = pd.read_csv(full_path, header=0)
data = data.sample(frac=1)

In [ ]:
# Drop rows with NaN values
data = data.dropna()

In [ ]:
input=data.iloc[:,0:16].values
output=data.iloc[:,16:20].values

In [ ]:
# Parameters
truncate_length = 3
truncate_length_pixels = truncate_length / 0.75

# Define Gaussian distribution parameters
mean = [12, 12]
covariance = np.eye(2) * 4 # modify it to 2, if target distribution needs to be sharper
distribution = multivariate_normal(mean=mean, cov=covariance)

# Define circular integration limits in polar coordinates
radius = truncate_length_pixels

# Define integrand function for integration
def integrand(r, theta):
    x = mean[0] + r * np.cos(theta)
    y = mean[1] + r * np.sin(theta)
    return distribution.pdf([x, y]) * r

# Perform double integration to get area under the curve
area_under_curve, error = dblquad(integrand, 0, 2*np.pi, lambda r: 0, lambda r: radius)

print("Area under the curve within the circle:", area_under_curve)

# Gaussian force map generation
num_rows = output.shape[0]
output_forcemap_gaussian_normalized = []

for i in range(num_rows):
    # Create a 25x25 array filled with zeros
    array_25x25 = np.zeros((25, 25))

    # Calculate the indices
    indices = np.column_stack(((output[i, 0] / -0.75).astype(int), (output[i, 1] / -0.75).astype(int)))

    # Create 2D Gaussian distribution centered at the calculated indices for output[i, 2]
    mean = indices[0, :]
    distribution = multivariate_normal(mean=mean, cov=covariance)

    # Create a grid of points to evaluate the Gaussian distribution
    x, y = np.meshgrid(np.arange(25), np.arange(25))
    grid = np.column_stack((x.flatten(), y.flatten()))

    # Evaluate the Gaussian distribution at each point in the grid
    gaussian_values = distribution.pdf(grid).reshape(25, 25)

    if output[i, 2] < -0.02:
        # Multiply Gaussian values by output[i, 2] and add to array_25x25
        curve_values = gaussian_values * output[i, 2]

        # Truncate the Gaussian distribution
        distance_from_center = np.sqrt((x - mean[0])**2 + (y - mean[1])**2)
        curve_values[distance_from_center > truncate_length_pixels] = 0

        # Normalize the truncated curve
        curve_values = (curve_values / area_under_curve)

        # Add normalized curve to array_25x25
        array_25x25 += curve_values

    # Append resulting array_25x25 to the list
    output_forcemap_gaussian_normalized.append(array_25x25)

# Convert list to NumPy array
output_forcemap_gaussian_normalized = np.array(output_forcemap_gaussian_normalized)

In [ ]:
# Flipping the force map along Y axis to make sure that the input and output have have indentations at the same location and are well aligned
output_forcemap_gaussian_flipped = np.flip(output_forcemap_gaussian_normalized, axis=2)

In [ ]:
# Example
# Visualize the flipped force map
i = 100

plt.imshow(output_forcemap_gaussian_flipped[i], cmap='viridis')
plt.colorbar()
plt.show()

display(output[i])

sum_of_elements = np.sum(output_forcemap_gaussian_normalized[i])

print("Sum of elements:", sum_of_elements)

In [ ]:
output_forcemap_gaussian_flipped_copy = output_forcemap_gaussian_flipped.copy()

In [ ]:
# Defining inputs and output for model training
x_train = torch.Tensor(input)
y_train = torch.Tensor(output_forcemap_gaussian_flipped_copy)
y_train_classifier = torch.Tensor(output[:, 3])

In [ ]:
# Reorganize input
x_train = x_train.view((x_train.shape[0], 4, 4, ))

In [ ]:
# Linear interpolation for the single channel
upscaled_x = F.interpolate(x_train.unsqueeze(1), scale_factor=25/4, mode='bicubic')
upscaled_x = upscaled_x.squeeze(1)

# Print the sizes for verification
print("Original Size:", x_train.size())
print("Upscaled (Linear) Size:", upscaled_x.size())

In [ ]:
plt.imshow(x_train[112].cpu(), cmap='viridis', origin='upper')
plt.colorbar()
plt.show()

# Visualize the upscaled input
plt.imshow(upscaled_x[112].cpu(), cmap='viridis')
plt.colorbar()
plt.show()

In [ ]:
mean_input_train = upscaled_x.mean().item()
std_input_train = upscaled_x.std().item()
input_transform = transforms.Normalize(mean=[mean_input_train], std=[std_input_train])
upscaled_x_train_normalized = input_transform(upscaled_x)
mean_output_train = y_train.mean().item()
std_output_train = y_train.std().item()
output_transform = transforms.Normalize(mean=[mean_output_train], std=[std_output_train])
y_train_normalized = output_transform(y_train)
# Moving to GPU
upscaled_x_train_normalized = upscaled_x_train_normalized.to(device)
y_train_normalized = y_train_normalized.to(device)
y_train_classifier = y_train_classifier.to(device)
# Create TensorDataset with normalized inputs and outputs
train_dataset = TensorDataset(upscaled_x_train_normalized, y_train_normalized, y_train_classifier)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=True)

In [ ]:
class SRCNNWithBinaryOutput(nn.Module):
    def __init__(self):
        super(SRCNNWithBinaryOutput, self).__init__()

        self.layer1 = nn.Conv2d(1, 64, kernel_size=9, padding=4)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer2 = nn.Conv2d(64, 32, kernel_size=1, padding=0)
        self.bn2 = nn.BatchNorm2d(32)
        self.layer3 = nn.Conv2d(32, 1, kernel_size=5, padding=2)

        # Additional FC layers
        self.fc1 = nn.Linear(1 * 25 * 25, 512)
        self.bn3 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 1024) 
        self.bn4 = nn.BatchNorm1d(1024)  
        self.fc3 = nn.Linear(1024, 512)  
        self.bn5 = nn.BatchNorm1d(512)  
        self.fc4 = nn.Linear(512, 25 * 25)

        # Binary output layer
        self.fc_binary = nn.Linear(25 * 25, 1)

    def forward(self, x):
        x = F.relu(self.bn1(self.layer1(x)))
        x = F.relu(self.bn2(self.layer2(x)))
        x = self.layer3(x)
        
        # Flatten the output before FC layers
        x_flat = x.view(x.size(0), -1)
        
        # FC layers for image prediction
        x_img = F.relu(self.bn3(self.fc1(x_flat)))
        x_img = F.relu(self.bn4(self.fc2(x_img))) 
        x_img = F.relu(self.bn5(self.fc3(x_img)))  
        x_img = self.fc4(x_img)

        # Reshape to 25x25
        x_img = x_img.view(-1, 1, 25, 25)

        # FC layer for binary prediction
        x_binary = torch.sigmoid(self.fc_binary(x_flat))

        return x_img, x_binary

In [ ]:
# Instantiate the model
model = SRCNNWithBinaryOutput()

# Input tensor
input_tensor = torch.randn((8, 1, 25, 25))

# Forward pass
output_img, output_binary = model(input_tensor)

# Print the shapes of the outputs
print("Output Image Shape:", output_img.shape)
print("Output Binary Shape:", output_binary.shape)

In [ ]:
# Define the target device (GPU if available, otherwise CPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Instantiate the CNNRegression model
model = SRCNNWithBinaryOutput()

# Move the model to the specified device
model = model.to(device)

model.cuda

In [ ]:
# Parallelizing the training of a neural network across multiple GPUs
data_parallel_model = torch.nn.DataParallel(model)

In [ ]:
# Define the criterion and optimizer
criterion_img = nn.MSELoss()  # MSE loss for image prediction
criterion_binary = nn.BCELoss()  # Binary Cross Entropy loss for binary prediction

# Hyperparameter to control the weight of loss_img and loss_binary in the total loss
alpha = 0.5  # adjust this value based on your preference

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# Train the model
for epoch in range(10):
    for i, (inputs, targets, targets_classifier) in enumerate(train_loader):

        # Zero the gradients
        optimizer.zero_grad()        

        # Forward pass
        outputs_img, outputs_binary = data_parallel_model(inputs.unsqueeze(1))

        outputs_img_squeezed = outputs_img.squeeze()

        # Compute MSE loss for the image prediction
        loss_img = criterion_img(outputs_img_squeezed, targets)

        # Compute Binary Cross Entropy loss for the binary prediction
        loss_binary = criterion_binary(outputs_binary, targets_classifier.unsqueeze(1).float())

        # Total loss is a weighted sum of image prediction loss and binary prediction loss
        loss_total = alpha * loss_img + (1 - alpha) * loss_binary

        # Backward pass and optimization
        loss_total.backward()
        optimizer.step()

        # Print training statistics
        if (i + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{10}], Step [{i + 1}/{len(train_loader)}], '
                  f'Loss Image: {loss_img.item():.4f}, Loss Binary: {loss_binary.item():.4f}, '
                  f'Weighted Total Loss: {loss_total.item():.4f}')
            

## Save the trained model

In [ ]:
save_path = os.path.join(f'../../../Data/Trained Models/Force Map - Trained Model/SRCNNWithBinaryOutput_{datetime.now().strftime("%Y%m%d-%H_%M")}.pth')
torch.save({
    'model_state_dict': data_parallel_model.state_dict(),
    'model_class': SRCNNWithBinaryOutput,
}, save_path)
time.sleep(5)

In [ ]:
# Performance check on test dataset

#importing the dataset
data_test = pd.read_csv('../../../Data/Datasets/2. Sensor 2/Test Dataset/test_ml_dataset_sensorv2_02-09-2024_09-33.txt')
data_test = data_test.sample(frac=1)
input_test =data_test.iloc[:,0:16].values
output_test=data_test.iloc[:,16:20].values

In [ ]:
# Parameters
truncate_length = 3
truncate_length_pixels = truncate_length / 0.75

# Define Gaussian distribution parameters
mean = [12, 12]
covariance = np.eye(2) * 4
distribution = multivariate_normal(mean=mean, cov=covariance)

# Define circular integration limits in polar coordinates
radius = truncate_length_pixels

# Define integrand function for integration
def integrand(r, theta):
    x = mean[0] + r * np.cos(theta)
    y = mean[1] + r * np.sin(theta)
    return distribution.pdf([x, y]) * r

# Perform double integration to get area under the curve
area_under_curve, error = dblquad(integrand, 0, 2*np.pi, lambda r: 0, lambda r: radius)

print("Area under the curve within the circle:", area_under_curve)

# Gaussian force map generation
num_rows = output_test.shape[0]
output_test_forcemap_gaussian_normalized = []

for i in range(num_rows):
    # Create a 25x25 array filled with zeros
    array_25x25 = np.zeros((25, 25))

    # Calculate the indices
    indices = np.column_stack(((output_test[i, 0] / -0.75).astype(int), (output_test[i, 1] / -0.75).astype(int)))

    # Create 2D Gaussian distribution centered at the calculated indices for output[i, 2]
    mean = indices[0, :]
    distribution = multivariate_normal(mean=mean, cov=covariance)

    # Create a grid of points to evaluate the Gaussian distribution
    x, y = np.meshgrid(np.arange(25), np.arange(25))
    grid = np.column_stack((x.flatten(), y.flatten()))

    # Evaluate the Gaussian distribution at each point in the grid
    gaussian_values = distribution.pdf(grid).reshape(25, 25)

    if output_test[i, 2] < -0.02:
        # Multiply Gaussian values by output[i, 2] and add to array_25x25
        curve_values = gaussian_values * output_test[i, 2]

        # Truncate the Gaussian distribution
        distance_from_center = np.sqrt((x - mean[0])**2 + (y - mean[1])**2)
        curve_values[distance_from_center > truncate_length_pixels] = 0

        # Normalize the truncated curve
        curve_values = (curve_values / area_under_curve)

        # Add normalized curve to array_25x25
        array_25x25 += curve_values

    # Append resulting array_25x25 to the list
    output_test_forcemap_gaussian_normalized.append(array_25x25)

In [ ]:
output_test_forcemap_gaussian_normalized = np.array(output_test_forcemap_gaussian_normalized)
output_test_forcemap_gaussian_flipped = np.flip(output_test_forcemap_gaussian_normalized, axis=2)
output_test_forcemap_gaussian_flipped_copy = output_test_forcemap_gaussian_flipped.copy()
x_test = torch.Tensor(input_test)
y_test = torch.Tensor(output_test_forcemap_gaussian_flipped_copy)
x_test = x_test.view((x_test.shape[0], 4, 4, ))
y_test_classifier = torch.Tensor(output_test[:, 3])
upscaled_x = F.interpolate(x_test.unsqueeze(1), scale_factor=25/4, mode='bicubic')
upscaled_x = upscaled_x.squeeze(1)

In [ ]:
upscaled_x_test = upscaled_x.to(device)

input_transform = transforms.Normalize(mean=[mean_input_train], std=[std_input_train])
upscaled_x_test_normalized = input_transform(upscaled_x_test)

upscaled_x_test_normalized = upscaled_x_test_normalized.to(device)
y_test = y_test.to(device)
y_test_classifier = y_test_classifier.to(device)
target_outputs_loc = (torch.Tensor(output_test[:, 0:3])).to(device)

# Include original target locations as part of the dataset
test_dataset = TensorDataset(upscaled_x_test_normalized, y_test, y_test_classifier, target_outputs_loc)

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, drop_last=True)
test_loader_size = len(test_loader)
print("Size of test_loader:", test_loader_size)
test_iterator = iter(test_loader)

In [ ]:
print("upscaled_x_test_normalized shape:", upscaled_x_test_normalized.shape)
print("y_test shape:", y_test.shape)
print("y_test_classifier shape:", y_test_classifier.shape)
print("output_test shape:", torch.Tensor(output_test[:, 0:2]).shape)

In [ ]:
# Print the sizes for verification
print("Original Size:", x_test.size())
print("Upscaled (Linear) Size:", upscaled_x.size())

In [ ]:
dataframe = pd.DataFrame()
m = nn.Sigmoid()
inverse_output_transform = transforms.Normalize(mean=[-mean_output_train/std_output_train], std=[1/std_output_train])
with torch.no_grad():
    for i, (inputs, targets, targets_classifier, original_target_locations) in enumerate(test_loader): 
        outputs_img, outputs_binary = model(inputs.unsqueeze(1))
        outputs_img_squeezed = outputs_img.squeeze()
        outputs_img_squeezed = inverse_output_transform(outputs_img_squeezed)
        outputs_img_squeezed_revert = -1*outputs_img_squeezed
        bottomk_values, bottomk_indices = torch.topk(outputs_img_squeezed_revert.view(8, -1), k=25*25-49, dim=1, largest=False)
        mask = torch.ones_like(outputs_img_squeezed.view(8, -1))
        mask.scatter_(1, bottomk_indices, 0)
        output_map_least_49 = outputs_img_squeezed * mask.view(8, 25, 25)
        total_normal_vector_output_49 = torch.sum(output_map_least_49, dim=(1, 2))
        total_normal_vector_target = torch.sum(targets, dim=(1, 2))
        touch_classification_output = outputs_binary

        # flipping the output and target maps back
        output_map_least_49 = torch.flip(output_map_least_49, dims=[2])
        targets = torch.flip(targets, dims=[2])
        # List to store mean locations for each image
        mean_locations_prediction = []

        # Iterate through each image
        for i in range(output_map_least_49.shape[0]):
            # Get the nonzero indices for the current image
            nonzero_indices = torch.nonzero(output_map_least_49[i])

            # Compute the mean location for the current image
            mean_location = torch.mean(nonzero_indices.float(), dim=0).int()

            # Extract the mean x and y coordinates
            mean_x, mean_y = mean_location[0].item(), mean_location[1].item()

            # Divide mean x and y by -0.75 for "Location_prediction"
            location_prediction_x, location_prediction_y = mean_x * -0.75, mean_y * -0.75

            # Append the location prediction to the list
            location_prediction = (location_prediction_x, location_prediction_y)
            mean_locations_prediction.append(location_prediction)

        weighted_mean_locations_prediction = []

        # Iterate through each image
        for i in range(output_map_least_49.shape[0]):
            # Get the non-zero indices and values for the current image
            nonzero_indices = torch.nonzero(output_map_least_49[i])
            nonzero_values = output_map_least_49[i][nonzero_indices[:, 0], nonzero_indices[:, 1]]

            # Compute the weighted mean location for the current image
            weighted_mean_location = torch.sum(nonzero_indices.float() * nonzero_values.unsqueeze(1), dim=0) / torch.sum(nonzero_values.unsqueeze(1))

            # Extract the weighted mean x and y coordinates
            weighted_mean_x, weighted_mean_y = weighted_mean_location[0].item(), weighted_mean_location[1].item()
            location_prediction_x, location_prediction_y = weighted_mean_x * -0.75, weighted_mean_y * -0.75

            # Round the values to two decimal places
            location_prediction_x = round(location_prediction_x, 2)
            location_prediction_y = round(location_prediction_y, 2)

            # Append the weighted mean location prediction to the list
            location_prediction = (location_prediction_x, location_prediction_y)
            weighted_mean_locations_prediction.append(location_prediction)

        # List to store mean locations for each image
        mean_locations_target = []
        mean_locations_target_calc = []

        # Iterate through each image
        for i in range(targets.shape[0]):
            # Get the nonzero indices for the current image
            nonzero_indices = torch.nonzero(targets[i])

            # Compute the mean location for the current image
            mean_location = torch.mean(nonzero_indices.float(), dim=0).int()

            # Extract the mean x and y coordinates
            mean_x_calc, mean_y_calc = mean_location[0].item(), mean_location[1].item()
            location_true_x, location_true_y = mean_x_calc * -0.75, mean_y_calc * -0.75

            # Append the location prediction to the list
            location_true_calc = (location_true_x, location_true_y)
            mean_locations_target_calc.append(location_true_calc)
        
        # Move tensors to CPU
        original_target_locations_cpu = original_target_locations[:, [1, 0]].to('cpu')
        mean_locations_target_tensor_calc_cpu = torch.tensor(mean_locations_target_calc).to('cpu')
        total_normal_vector_target_tensor = total_normal_vector_target.unsqueeze(1).to('cpu')
        targets_classifier_tensor = targets_classifier.unsqueeze(1).to('cpu')
        mean_locations_prediction_tensor = torch.tensor(mean_locations_prediction).to('cpu')
        weighted_mean_locations_prediction_tensor = torch.tensor(weighted_mean_locations_prediction).to('cpu')
        total_normal_vector_output_49_tensor = total_normal_vector_output_49.unsqueeze(1).to('cpu')
        touch_classification_output_tensor = touch_classification_output.to('cpu')

        # Concatenate tensors
        data_row = torch.cat((original_target_locations_cpu, mean_locations_target_tensor_calc_cpu, total_normal_vector_target_tensor, targets_classifier_tensor, mean_locations_prediction_tensor, weighted_mean_locations_prediction_tensor, total_normal_vector_output_49_tensor, touch_classification_output_tensor), dim=1)
        dataframe = pd.concat([dataframe,pd.DataFrame(data_row.data.cpu())])

# Save the Pandas DataFrame to an Excel file
dataframe.to_excel("../../../Data/Results/Results/Force_Map_Estimation_Results/test_Error_inference.xlsx", index=False)